In [20]:
import re

def read_fts(fts_file):
    parsing = False
    headers = [] 
    data_dict = {}
    header_lines = []
    column_widths = []
    expected_shape = {"rows": None, "columns": None}

    with open(fts_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    # Extract row/column counts from the top of the file
    for line in lines:
        if "Columns in File" in line:
            match = re.search(r"Columns in File:\s*(\d+)", line)
            if match:
                expected_shape["columns"] = int(match.group(1))
        if "Exact File Quantity" in line:
            match = re.search(r"Exact File Quantity.*?(\d[\d,]*)", line)
            if match:
                expected_shape["rows"] = int(match.group(1).replace(",", ""))

    # Find the header lines and the start of the table
    for i, line in enumerate(lines):
        line = line.rstrip()
        if '----' in line:
            parsing = True
            if i > 2:
                header_lines = [l.rstrip() for l in lines[i-3:i]]
            column_widths = [len(match.group()) for match in re.finditer(r'-+', line)]
            break

    # Extract column headers
    start = 0
    for width in column_widths:
        column_header = ' '.join(line[start:start+width].strip() for line in header_lines).strip()
        headers.append(column_header)
        start += width + 1

    for header in headers:
        data_dict[header] = []

    # Parse table rows
    for line in lines[i+1:]:
        line = line.rstrip()
        if not line or "Note:" in line:
            break
        start = 0
        row_values = []
        for width in column_widths:
            row_values.append(line[start:start+width].strip())
            start += width + 1
        if len(row_values) == len(headers):
            for header, value in zip(headers, row_values):
                data_dict[header].append(value)

    return {"schema": data_dict, "expected_shape": expected_shape}


In [21]:
fts_data = read_fts("/n/dominici_nsaph_l3/Lab/data/ci3_d_medicare/original_data/cms_medicare/data/4334/2011/mbsf_ab_summary_res000017155_req004334_2011.fts")

In [22]:
expected_schema = fts_data['schema']

In [23]:
import duckdb
from pathlib import Path
from rich import print as rprint
from rich.pretty import Pretty
from rich.panel import Panel

def run_duckdb_data_quality_checks(base_path: str, report_path: str, fts_dict: dict, metadata: dict = None):
    """
    base_path: directory containing multiple Parquet chunks for one table
    fts_dict: parsed .fts dictionary (each row describes a column)
    metadata: dict with 'rows' and 'columns' from FTS file header, if available
    """
    base_path = Path(base_path)
    parquet_glob = str(base_path / "*.parquet")

    con = duckdb.connect()
    con.execute(f"CREATE OR REPLACE VIEW df AS SELECT * FROM '{parquet_glob}'")
    report = {}

    # ---- SHAPE CHECK ----
    actual_rows = con.execute("SELECT COUNT(*) FROM df").fetchone()[0]
    actual_cols = len(con.execute("PRAGMA table_info('df')").fetchall())
    shape_report = {"actual_rows": actual_rows, "actual_columns": actual_cols}

    if metadata:
        expected_rows = metadata.get("rows")
        expected_cols = metadata.get("columns")
        if expected_rows is not None:
            shape_report["expected_rows"] = expected_rows
            shape_report["row_mismatch"] = (actual_rows != expected_rows)
        if expected_cols is not None:
            shape_report["expected_columns"] = expected_cols
            shape_report["column_mismatch"] = (actual_cols != expected_cols)

    report["shape_check"] = shape_report

    # ---- COLUMN CHECKS ----
    # Extract expected column names from FTS (list of strings)
    expected_colnames = [str(x).strip() for x in fts_dict.get("Field Short Name", []) if x and str(x).strip()]
    
    # Get actual column names from DuckDB
    actual_columns = [row[1] for row in con.execute("PRAGMA table_info('df')").fetchall()]
    
    # Compare expected and actual
    missing_columns = [col for col in expected_colnames if col not in actual_columns]
    unexpected_columns = [col for col in actual_columns if col not in expected_colnames]
    report["missing_columns"] = missing_columns
    report["unexpected_columns"] = unexpected_columns

    # ---- COLUMN-SPECIFIC CHECKS ----
    column_checks = {}

    for col in expected_colnames:
        if col not in actual_columns:
            continue

        col_report = {}

        # Null check
        col_report["missing_values"] = con.execute(
            f"SELECT COUNT(*) FROM df WHERE {col} IS NULL"
        ).fetchone()[0]

        # TODO: Extend with more FTS-based logic if needed
        column_checks[col] = col_report

    report["column_checks"] = column_checks

    # ---- DUPLICATES ----
    column_list = ', '.join(actual_columns)

    # Modify the query to group by all columns and count duplicates
    try:
        query = f"""
            SELECT SUM(duplicate_count) AS duplicate_count
            FROM (
                SELECT COUNT(*) AS duplicate_count
                FROM df
                GROUP BY {column_list}
                HAVING COUNT(*) > 1
            ) AS duplicate_counts
        """
        dupe_count = con.execute(query).fetchone()[0]
        report["duplicate_rows"] = int(dupe_count) if dupe_count is not None else 0
    except Exception as e:
        report["duplicate_rows"] = f"Error checking duplicates: {e}"


    # ---- NUMERIC SUMMARY ----
    numeric_cols = con.execute("""
        SELECT column_name 
        FROM information_schema.columns 
        WHERE table_name = 'df' AND data_type IN ('INTEGER', 'BIGINT', 'DOUBLE', 'FLOAT')
    """).fetchall()
    numeric_summary = {}
    for col in [row[0] for row in numeric_cols]:
        try:
            stats = con.execute(f"""
                SELECT 
                    MIN({col}), 
                    MAX({col}), 
                    AVG({col}), 
                    CASE 
                        WHEN COUNT({col}) > 1 THEN STDDEV_POP({col})
                        ELSE NULL
                    END
                FROM df
            """).fetchone()

            numeric_summary[col] = {
                "min": stats[0], 
                "max": stats[1], 
                "mean": stats[2], 
                "stddev": stats[3]
            }
        except Exception as e:
            numeric_summary[col] = {
                "error": f"Could not summarize {col}: {e}"
            }

    report["numeric_summary"] = numeric_summary

    # ---- TOP CATEGORIES ----
    string_cols = con.execute("""
        SELECT column_name 
        FROM information_schema.columns 
        WHERE table_name = 'df' AND data_type = 'VARCHAR'
    """).fetchall()
    top_categories = {}
    for col in [row[0] for row in string_cols]:
        counts = con.execute(
            f"SELECT {col}, COUNT(*) FROM df GROUP BY {col} ORDER BY COUNT(*) DESC LIMIT 5"
        ).fetchall()
        top_categories[col] = {val: count for val, count in counts if val is not None}
    report["top_categories"] = top_categories

    # ---- PRETTY REPORT ----
    rprint(Panel.fit("✅ [bold green]Data Quality Report"))
    for section, content in report.items():
        rprint(Panel(Pretty(content, indent_guides=True), title=f"[bold cyan]{section}"))
        
    with open(report_path, "w") as f:
        json.dump(report, f, indent=4)
        rprint(f"[bold yellow]Report saved to {report_path}")

    return report




In [26]:
fts_data = read_fts("/n/dominici_nsaph_l3/Lab/data/ci3_d_medicare/original_data/cms_medicare/data/10411/2017/mbsf_abcd_summary_res000017155_req010411_2017.fts")

expected_schema = fts_data['schema']

report = run_duckdb_data_quality_checks(
    base_path="/n/dominici_nsaph_l3/Lab/data/data_warehouse/dw_raw_medicare/2017/mbsf_abcd_summary_res000017155_req010411", 
    report_path="/n/dominici_nsaph_l3/Lab/data/data_warehouse/dw_raw_medicare/2017/mbsf_abcd_summary_res000017155_req010411/qc.json",
    fts_dict=expected_schema,
    metadata=fts_data["expected_shape"]
)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

╭────────────────────────╮
│ ✅ Data Quality Report │
╰────────────────────────╯

╭────────────────────────────────────────────────── shape_check ──────────────────────────────────────────────────╮
│ {                                                                                                               │
│ │   'actual_rows': 61405844,                                                                                    │
│ │   'actual_columns': 185,                                                                                      │
│ │   'expected_rows': 61405844,                                                                                  │
│ │   'row_mismatch': False,                                                                                      │
│ │   'expected_columns': 185,                                                                                    │
│ │   'column_mismatch': False                                                                                    │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────────── column_checks ─────────────────────────────────────────────────╮
│ {                                                                                                               │
│ │   'BENE_ID': {'missing_values': 0},                                                                           │
│ │   'RFRNC_YR': {'missing_values': 0},                                                                          │
│ │   'ENRL_SRC': {'missing_values': 0},                                                                          │
│ │   'SAMPLE_GROUP': {'missing_values': 49125557},                                                               │
│ │   'EFIVEPCT': {'missing_values': 58272831},                                                                   │
│ │   'CRNT_BIC': {'missing_values': 0},                                                                          │
│ │   'STATE_CD': {'missing_values': 2359},                                                                       │
│ │   'CNTY_CD': {'missing_values': 0},                                                                           │
│ │   'ZIP_CD': {'missing_values': 0},                                                                            │
│ │   'STATE_CNTY_FIPS_CD_01': {'missing_values': 3519052},                                                       │
│ │   'STATE_CNTY_FIPS_CD_02': {'missing_values': 3443409},                                                       │
│ │   'STATE_CNTY_FIPS_CD_03': {'missing_values': 3327403},                                                       │
│ │   'STATE_CNTY_FIPS_CD_04': {'missing_values': 3239791},                                                       │
│ │   'STATE_CNTY_FIPS_CD_05': {'missing_values': 3132887},                                                       │
│ │   'STATE_CNTY_FIPS_CD_06': {'missing_values': 2997223},                                                       │
│ │   'STATE_CNTY_FIPS_CD_07': {'missing_values': 2837495},                                                       │
│ │   'STATE_CNTY_FIPS_CD_08': {'missing_values': 2687894},                                                       │
│ │   'STATE_CNTY_FIPS_CD_09': {'missing_values': 2533671},                                                       │
│ │   'STATE_CNTY_FIPS_CD_10': {'missing_values': 2386027},                                                       │
│ │   'STATE_CNTY_FIPS_CD_11': {'missing_values': 2267387},                                                       │
│ │   'STATE_CNTY_FIPS_CD_12': {'missing_values': 2140261},                                                       │
│ │   'AGE': {'missing_values': 0},                                                                               │
│ │   'BENE_DOB': {'missing_values': 0},                                                                          │
│ │   'V_DOD_SW': {'missing_values': 59114306},                                                                   │
│ │   'DEATH_DT': {'missing_values': 59113095},                                                                   │
│ │   'SEX': {'missing_values': 0},                                                                               │
│ │   'RACE': {'missing_values': 0},                                                                              │
│ │   'RTI_RACE_CD': {'missing_values': 0},                                                                       │
│ │   'COVSTART': {'missing_values': 0},                                                                          │
│ │   'OREC': {'missing_values': 0},                                                                              │
│ │   'CREC': {'missing_values': 0},                                                                              │
│ │   'ESRD_IND': {'missing_values': 0},                                                                          │
│ │   'MDCR_STUS_CD_01': {'missing_values': 11031},     

╭──────────────────────────────────────────────── duplicate_rows ─────────────────────────────────────────────────╮
│ 0                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── numeric_summary ────────────────────────────────────────────────╮
│ {                                                                                                               │
│ │   'RFRNC_YR': {'min': 2017.0, 'max': 2017.0, 'mean': 2017.0, 'stddev': 0.0},                                  │
│ │   'AGE': {'min': 0.0, 'max': 115.0, 'mean': 71.35203992961972, 'stddev': 11.52289628179419},                  │
│ │   'A_MO_CNT': {'min': 0.0, 'max': 12.0, 'mean': 11.37244696123711, 'stddev': 2.183643451581167},              │
│ │   'B_MO_CNT': {'min': 0.0, 'max': 12.0, 'mean': 10.435358790932016, 'stddev': 3.737617014490667},             │
│ │   'BUYIN_MO': {'min': 0.0, 'max': 12.0, 'mean': 1.949746737460363, 'stddev': 4.317106898667654},              │
│ │   'HMO_MO': {'min': 0.0, 'max': 12.0, 'mean': 3.869635632725771, 'stddev': 5.506668166624101},                │
│ │   'PTD_MO': {'min': 0.0, 'max': 12.0, 'mean': 8.356843088094351, 'stddev': 5.323493075133532},                │
│ │   'RDS_MO': {'min': 0.0, 'max': 12.0, 'mean': 0.3217621273962133, 'stddev': 1.9063643551153535},              │
│ │   'DUAL_MO': {'min': 0.0, 'max': 12.0, 'mean': 2.091124225896154, 'stddev': 4.427806257090102}                │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── top_categories ─────────────────────────────────────────────────╮
│ {                                                                                                               │
│ │   'BENE_ID': {                                                                                                │
│ │   │   'lllllllllllll7o': 1,                                                                                   │
│ │   │   'lllllllllllll47': 1,                                                                                   │
│ │   │   'llllllllllll007': 1,                                                                                   │
│ │   │   'lllllllllllllOU': 1,                                                                                   │
│ │   │   'lllllllllllll07': 1                                                                                    │
│ │   },                                                                                                          │
│ │   'ENRL_SRC': {'CME': 61405844},                                                                              │
│ │   'SAMPLE_GROUP': {'15': 9206825, '04': 2458719, '01': 614743},                                               │
│ │   'EFIVEPCT': {'Y': 3133013},                                                                                 │
│ │   'CRNT_BIC': {'A': 50954120, 'D': 3044337, 'B': 1961714, 'M': 1134992, 'T': 1098102},                        │
│ │   'STATE_CD': {'05': 6270383, '10': 4491602, '45': 4090206, '33': 3654994, '39': 2765819},                    │
│ │   'CNTY_CD': {'200': 2009966, '010': 1655915, '000': 1491981, '020': 1481600, '060': 1443912},                │
│ │   'ZIP_CD': {'99999': 479767, '32162': 41902, '08759': 22328, '85351': 20756, '85375': 20478},                │
│ │   'STATE_CNTY_FIPS_CD_01': {'06037': 1424005, '17031': 797451, '04013': 633207, '48201': 501778},             │
│ │   'STATE_CNTY_FIPS_CD_02': {'06037': 1425121, '17031': 797627, '04013': 635382, '48201': 502899},             │
│ │   'STATE_CNTY_FIPS_CD_03': {'06037': 1427713, '17031': 798625, '04013': 636929, '48201': 504202},             │
│ │   'STATE_CNTY_FIPS_CD_04': {'06037': 1430017, '17031': 799232, '04013': 638011, '48201': 505342},             │
│ │   'STATE_CNTY_FIPS_CD_05': {'06037': 1432177, '17031': 799877, '04013': 639499, '48201': 506522},             │
│ │   'STATE_CNTY_FIPS_CD_06': {'06037': 1435320, '17031': 801252, '04013': 641292, '48201': 507950},             │
│ │   'STATE_CNTY_FIPS_CD_07': {'06037': 1439387, '17031': 803138, '04013': 643123, '48201': 509965},             │
│ │   'STATE_CNTY_FIPS_CD_08': {'06037': 1442141, '17031': 804028, '04013': 645027, '48201': 511988},             │
│ │   'STATE_CNTY_FIPS_CD_09': {'06037': 1445816, '17031': 805561, '04013': 647126, '48201': 513674},             │
│ │   'STATE_CNTY_FIPS_CD_10': {'06037': 1448852, '17031': 806544, '04013': 649683, '48201': 515344},             │
│ │   'STATE_CNTY_FIPS_CD_11': {'06037': 1451590, '17031': 807568, '04013': 651714, '48201': 516915},             │
│ │   'STATE_CNTY_FIPS_CD_12': {'06037': 1454967, '17031': 808403, '04013': 653931, '48201': 518461},             │
│ │   'V_DOD_SW': {'V': 2291538},                                                                                 │
│ │   'SEX': {'2': 33328860, '1': 28076972, '0': 12},                                                             │
│ │   'RACE': {'1': 48838804, '2': 6570168, '5': 1816292, '4': 1486333, '3': 1291493},                            │
│ │   'RTI_RACE_CD': {'1': 45524762, '2': 6420603, '5': 5683153, '4': 2023348, '0': 950534},                      │
│ │   'OREC': {'0': 46820126, '1': 14239782, '2': 196473, '3': 149463},                                           │
│ │   'CREC': {'0': 52170944, '1': 9083167, '2': 114312, '3': 37421},                                             │
│ │   'ESRD_IND': {'0': 60820041, 'Y': 585803},         

Report saved to 
/n/dominici_nsaph_l3/Lab/data/data_warehouse/dw_raw_medicare/2017/mbsf_abcd_summary_res000017155_req010411/qc.json

In [11]:
# data quality checks for the parsed sas files: 

import duckdb
from pathlib import Path
from rich import print as rprint
from rich.panel import Panel
from rich.pretty import Pretty

def sas_qc(base_path: str):
    """
    Perform basic data quality checks on a directory of Parquet chunks,
    without needing an external schema file.

    Parameters:
    - base_path (str): Directory containing multiple Parquet chunks.

    Returns:
    - dict: Summary report with basic data quality checks.
    """
    base_path = Path(base_path)
    parquet_glob = str(base_path / "*.parquet")

    con = duckdb.connect()
    con.execute(f"CREATE OR REPLACE VIEW df AS SELECT * FROM '{parquet_glob}'")
    report = {}

    # ---- SHAPE CHECK ----
    actual_rows = con.execute("SELECT COUNT(*) FROM df").fetchone()[0]
    actual_cols = len(con.execute("PRAGMA table_info('df')").fetchall())
    report["shape"] = {"rows": actual_rows, "columns": actual_cols}

    # ---- COLUMN METADATA ----
    actual_columns = [row[1] for row in con.execute("PRAGMA table_info('df')").fetchall()]
    report["columns"] = actual_columns

    # ---- NULL VALUE CHECKS ----
    null_counts = {}
    for col in actual_columns:
        count = con.execute(f"SELECT COUNT(*) FROM df WHERE {col} IS NULL").fetchone()[0]
        null_counts[col] = count
    report["null_counts"] = null_counts

    # ---- NUMERIC SUMMARY ----
    numeric_cols = con.execute("""
        SELECT column_name 
        FROM information_schema.columns 
        WHERE table_name = 'df' AND data_type IN ('INTEGER', 'BIGINT', 'DOUBLE', 'FLOAT')
    """).fetchall()
    numeric_summary = {}
    for col in [row[0] for row in numeric_cols]:
        try:
            stats = con.execute(f"""
                SELECT 
                    MIN({col}), 
                    MAX({col}), 
                    AVG({col}), 
                    CASE 
                        WHEN COUNT({col}) > 1 THEN STDDEV_POP({col})
                        ELSE NULL
                    END
                FROM df
            """).fetchone()

            numeric_summary[col] = {
                "min": stats[0], 
                "max": stats[1], 
                "mean": stats[2], 
                "stddev": stats[3]
            }
        except Exception as e:
            numeric_summary[col] = {
                "error": f"Could not summarize {col}: {e}"
            }

    report["numeric_summary"] = numeric_summary

    # ---- TOP CATEGORIES ----
    string_cols = con.execute("""
        SELECT column_name 
        FROM information_schema.columns 
        WHERE table_name = 'df' AND data_type = 'VARCHAR'
    """).fetchall()
    top_categories = {}
    for col in [row[0] for row in string_cols]:
        counts = con.execute(
            f"SELECT {col}, COUNT(*) FROM df GROUP BY {col} ORDER BY COUNT(*) DESC LIMIT 5"
        ).fetchall()
        top_categories[col] = {val: count for val, count in counts if val is not None}
    report["top_categories"] = top_categories

    # ---- DUPLICATE CHECK ----
    try:
        column_list = ', '.join(actual_columns)
        dupe_query = f"""
            SELECT SUM(duplicate_count) AS duplicate_count
            FROM (
                SELECT COUNT(*) AS duplicate_count
                FROM df
                GROUP BY {column_list}
                HAVING COUNT(*) > 1
            ) AS duplicate_counts
        """
        dupe_count = con.execute(dupe_query).fetchone()[0]
        report["duplicate_rows"] = int(dupe_count) if dupe_count is not None else 0
    except Exception as e:
        report["duplicate_rows"] = f"Error checking duplicates: {e}"

    # ---- PRINT REPORT ----
    rprint(Panel.fit("📊 [bold blue]Basic Data Quality Report"))
    for section, content in report.items():
        rprint(Panel(Pretty(content, indent_guides=True), title=f"[bold cyan]{section}"))

    return report


In [12]:
import json
from pathlib import Path

def run_qc_for_years(base_path: str, years: range):
    """
    Runs basic QC on all subdirectories under base_path/year/* for the given year range.

    Parameters:
    - base_path (str): Root path containing yearly folders.
    - years (range): Range of years (e.g., range(2016, 2020)).

    Saves output to <base_path>/<year>/<fileprefix>/qc_report.json
    """
    base_path = Path(base_path)
    results = {}

    for year in years:
        year_str = str(year)
        year_path = base_path / year_str
        if not year_path.exists():
            print(f"Skipping missing year: {year_str}")
            continue

        for subdir in sorted(year_path.iterdir()):
            if not subdir.is_dir():
                continue

            print(f"📂 Running QC for: {subdir}")
            try:
                report = sas_qc(subdir)
                results[str(subdir)] = report

                # Save report next to data
                out_file = subdir / "qc.json"
                with open(out_file, "w") as f:
                    json.dump(report, f, indent=2)
            except Exception as e:
                print(f"❌ Failed QC for {subdir}: {e}")
                results[str(subdir)] = {"error": str(e)}

    return results




In [13]:
run_qc_for_years(
    base_path="/n/dominici_nsaph_l3/Lab/data/data_warehouse/dw_raw_medicare",
    years=range(1999, 2011)
)


📂 Running QC for: /n/dominici_nsaph_l3/Lab/data/data_warehouse/dw_raw_medicare/1999/dnm


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

╭──────────────────────────────╮
│ 📊 Basic Data Quality Report │
╰──────────────────────────────╯

╭───────────────────────────────────────────────────── shape ─────────────────────────────────────────────────────╮
│ {'rows': 41095965, 'columns': 23}                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────────── null_counts ──────────────────────────────────────────────────╮
│ {                                                                                                               │
│ │   'BENE_ID': 0,                                                                                               │
│ │   'MEDPAR_ID': 0,                                                                                             │
│ │   'EQTBL_BIC_CD': 0,                                                                                          │
│ │   'BENE_AGE_CNT': 0,                                                                                          │
│ │   'BENE_SEX_CD': 0,                                                                                           │
│ │   'BENE_RACE_CD': 0,                                                                                          │
│ │   'BENE_MDCR_STUS_CD': 0,                                                                                     │
│ │   'BENE_RSDNC_SSA_STATE_CD': 0,                                                                               │
│ │   'BENE_RSDNC_SSA_CNTY_CD': 0,                                                                                │
│ │   'BENE_MLG_CNTCT_ZIP_CD': 2805,                                                                              │
│ │   'ADMSN_DAY_CD': 0,                                                                                          │
│ │   'BENE_DSCHRG_STUS_CD': 0,                                                                                   │
│ │   'GHO_PD_CD': 15838784,                                                                                      │
│ │   'PPS_IND_CD': 0,                                                                                            │
│ │   'ORG_NPI_NUM': 718,                                                                                         │
│ │   'PRVDR_NUM': 0,                                                                                             │
│ │   'PRVDR_NUM_SPCL_UNIT_CD': 17108658,                                                                         │
│ │   'SS_LS_SNF_IND_CD': 0,                                                                                      │
│ │   'STAY_FINL_ACTN_CLM_CNT': 0,                                                                                │
│ │   'LTST_CLM_ACRTN_DT': 0,                                                                                     │
│ │   'BENE_MDCR_BNFT_EXHST_DT': 74768,                                                                           │
│ │   'SNF_QUALN_FROM_DT': 17793253,                                                                              │
│ │   'SNF_QUALN_THRU_DT': 17793253,                                                                              │
│ │   'ADMSN_DT': 0,                                                                                              │
│ │   'DSCHRG_DT': 17793253,                                                                                      │
│ │   'CVRD_LVL_CARE_THRU_DT': 17793253,                                                                          │
│ │   'BENE_DEATH_DT': 17793253,                                                                                  │
│ │   'BENE_DEATH_DT_VRFY_CD': 13174945,                                                                          │
│ │   'INTRNL_USE_SSI_IND_CD': 16090231,                                                                          │
│ │   'INTRNL_USE_SSI_DAY_CNT': 0,                                                                                │
│ │   'LOS_DAY_CNT': 0,                                                                                           │
│ │   'OUTLIER_DAY_CNT': 0,                                                                                       │
│ │   'UTLZTN_DAY_CNT': 0,                              

╭──────────────────────────────────────────────── numeric_summary ────────────────────────────────────────────────╮
│ {                                                                                                               │
│ │   'BENE_AGE_CNT': {'min': 0.0, 'max': 116.0, 'mean': 73.80254077205556, 'stddev': 13.617430486636676},        │
│ │   'STAY_FINL_ACTN_CLM_CNT': {                                                                                 │
│ │   │   'min': 1.0,                                                                                             │
│ │   │   'max': 78.0,                                                                                            │
│ │   │   'mean': 1.148697093218424,                                                                              │
│ │   │   'stddev': 0.6422732068191204                                                                            │
│ │   },                                                                                                          │
│ │   'BENE_MDCR_BNFT_EXHST_DT': {                                                                                │
│ │   │   'error': 'Could not summarize BENE_MDCR_BNFT_EXHST_DT: Out of Range Error: STDDEV_POP is out of range!' │
│ │   },                                                                                                          │
│ │   'INTRNL_USE_SSI_DAY_CNT': {                                                                                 │
│ │   │   'min': 0.0,                                                                                             │
│ │   │   'max': 916.0,                                                                                           │
│ │   │   'mean': 0.46298892057568114,                                                                            │
│ │   │   'stddev': 2.9074807339379327                                                                            │
│ │   },                                                                                                          │
│ │   'LOS_DAY_CNT': {'min': 1.0, 'max': 3998.0, 'mean': 9.93704057375006, 'stddev': 22.051171095861687},         │
│ │   'OUTLIER_DAY_CNT': {'min': 0.0, 'max': 0.0, 'mean': 0.0, 'stddev': 0.0},                                    │
│ │   'UTLZTN_DAY_CNT': {'min': 0.0, 'max': 999.0, 'mean': 8.649945515864918, 'stddev': 13.293028019141524},      │
│ │   'TOT_COINSRNC_DAY_CNT': {                                                                                   │
│ │   │   'min': 0.0,                                                                                             │
│ │   │   'max': 570.0,                                                                                           │
│ │   │   'mean': 2.3771518339001867,                                                                             │
│ │   │   'stddev': 9.914070241388528                                                                             │
│ │   },                                                                                                          │
│ │   'BENE_LRD_USE_CNT': {'min': 0.0, 'max': 999.0, 'mean': 0.05462643621152355, 'stddev': 1.2674687015391184},  │
│ │   'BENE_PTA_COINSRNC_AMT': {                                                                                  │
│ │   │   'min': 0.0,                                                                                             │
│ │   │   'max': 571590.0,                                                                                        │
│ │   │   'mean': 373.4090421240006,                                                                              │
│ │   │   'stddev': 1610.6808762168505                                                                            │
│ │   },                                                                                                          │
│ │   'BENE_IP_DDCTBL_AMT': {                           

╭──────────────────────────────────────────────── top_categories ─────────────────────────────────────────────────╮
│ {                                                                                                               │
│ │   'BENE_ID': {                                                                                                │
│ │   │   'llllllloS8000lS': 103,                                                                                 │
│ │   │   'lllllllUOOl0Oo0': 94,                                                                                  │
│ │   │   'lllllllUO74UoXO': 88,                                                                                  │
│ │   │   'lllllllU787UoX4': 85,                                                                                  │
│ │   │   'lllllllUO0U8SoO': 81                                                                                   │
│ │   },                                                                                                          │
│ │   'MEDPAR_ID': {                                                                                              │
│ │   │   'llllllX7O44UolS': 1,                                                                                   │
│ │   │   'llllllX7XUlolo0': 1,                                                                                   │
│ │   │   'llllllX7U074XU8': 1,                                                                                   │
│ │   │   'llllllXO7UOXooO': 1,                                                                                   │
│ │   │   'llllllX70o40084': 1                                                                                    │
│ │   },                                                                                                          │
│ │   'EQTBL_BIC_CD': {'A': 14748984, 'B': 2415162, 'C1': 191451, '14': 103000, '10': 93443},                     │
│ │   'BENE_SEX_CD': {'2': 10075437, '1': 7717785, '0': 31},                                                      │
│ │   'BENE_RACE_CD': {'1': 14566007, '2': 2274806, '5': 397503, '3': 218564, '4': 195888},                       │
│ │   'BENE_MDCR_STUS_CD': {'10': 14101471, '20': 2926898, '11': 380279, '21': 243553, '31': 141052},             │
│ │   'BENE_RSDNC_SSA_STATE_CD': {'05': 1402007, '10': 1322129, '45': 1180685, '33': 1161681, '39': 938690},      │
│ │   'BENE_RSDNC_SSA_CNTY_CD': {'200': 532627, '010': 458142, '020': 404776, '000': 374062, '060': 360020},      │
│ │   'BENE_MLG_CNTCT_ZIP_CD': {'08759': 11047, '08757': 9099, '32162': 7598, '11235': 7583, '85351': 7405},      │
│ │   'ADMSN_DAY_CD': {'3': 3044788, '4': 3038470, '5': 2850099, '7': 2790017, '6': 2783914},                     │
│ │   'BENE_DSCHRG_STUS_CD': {'A': 16672713, 'B': 588026, 'C': 532514},                                           │
│ │   'GHO_PD_CD': {'1': 1954469},                                                                                │
│ │   'PPS_IND_CD': {'2': 16394887, '0': 1398366},                                                                │
│ │   'ORG_NPI_NUM': {                                                                                            │
│ │   │   '1306938071': 51701,                                                                                    │
│ │   │   '1952476988': 33677,                                                                                    │
│ │   │   '1821007881': 31018,                                                                                    │
│ │   │   '1598744856': 28430,                                                                                    │
│ │   │   '1124074273': 28146                                                                                     │
│ │   },                                                                                                          │
│ │   'PRVDR_NUM': {'100007': 51701, '330101': 36997, '3

╭──────────────────────────────────────────────── duplicate_rows ─────────────────────────────────────────────────╮
│ 0                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

{'/n/dominici_nsaph_l3/Lab/data/data_warehouse/dw_raw_medicare/1999/dnm': {'shape': {'rows': 41095965,
   'columns': 23},
  'columns': ['STATE',
   'ZIPCODE',
   'DOB',
   'SEX',
   'RACE',
   'AGE',
   'ORIG_ENT',
   'CUR_ENT',
   'ESRD_IND',
   'MCSTATUS',
   'PRTATERM',
   'PRTBTERM',
   'MC_ENT',
   'HMOIND',
   'HICOVG',
   'SMICOVG',
   'HMOCOVG',
   'BUYCOVG',
   'DODFLAG',
   'BEF_DOD',
   'ENROLYR',
   'FIVE_PERCENT_FLAG',
   'Intbid'],
  'null_counts': {'STATE': 0,
   'ZIPCODE': 0,
   'DOB': 0,
   'SEX': 0,
   'RACE': 0,
   'AGE': 0,
   'ORIG_ENT': 0,
   'CUR_ENT': 0,
   'ESRD_IND': 0,
   'MCSTATUS': 0,
   'PRTATERM': 0,
   'PRTBTERM': 0,
   'MC_ENT': 0,
   'HMOIND': 0,
   'HICOVG': 0,
   'SMICOVG': 0,
   'HMOCOVG': 0,
   'BUYCOVG': 0,
   'DODFLAG': 38785745,
   'BEF_DOD': 0,
   'ENROLYR': 0,
   'FIVE_PERCENT_FLAG': 0,
   'Intbid': 0},
  'numeric_summary': {'ZIPCODE': {'min': 0.0,
    'max': 999999999.0,
    'mean': 480117218.7108296,
    'stddev': 295835210.87882584},
   'DO

In [9]:
#2007 medpar is empty. is this a parsing error or an error with the sas file itself? 

import duckdb
from pathlib import Path
base_path = "/n/dominici_nsaph_l3/Lab/data/data_warehouse/dw_raw_medicare/2007/medpar"
base_path = Path(base_path)
parquet_glob = str(base_path / "*.parquet")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE VIEW df AS SELECT * FROM '{parquet_glob}'")

# Fetch and display the first 500 rows
result = con.execute("SELECT * FROM df LIMIT 500").fetchdf()
print(result)


Empty DataFrame
Columns: [BENE_ID, EQTBL_BIC_CD, BENE_AGE_CNT, BENE_SEX_CD, BENE_RACE_CD, BENE_MDCR_STUS_CD, BENE_RSDNC_SSA_STATE_CD, BENE_RSDNC_SSA_CNTY_CD, BENE_MLG_CNTCT_ZIP_CD, ADMSN_DAY_CD, BENE_DSCHRG_STUS_CD, GHO_PD_CD, PPS_IND_CD, ORG_NPI_NUM, PRVDR_NUM, PRVDR_NUM_SPCL_UNIT_CD, SS_LS_SNF_IND_CD, STAY_FINL_ACTN_CLM_CNT, LTST_CLM_ACRTN_DT, BENE_MDCR_BNFT_EXHST_DT, SNF_QUALN_FROM_DT, SNF_QUALN_THRU_DT, ADMSN_DT, DSCHRG_DT, CVRD_LVL_CARE_THRU_DT, BENE_DEATH_DT, BENE_DEATH_DT_VRFY_CD, INTRNL_USE_SSI_IND_CD, INTRNL_USE_SSI_DAY_CNT, LOS_DAY_CNT, OUTLIER_DAY_CNT, UTLZTN_DAY_CNT, TOT_COINSRNC_DAY_CNT, BENE_LRD_USE_CNT, BENE_PTA_COINSRNC_AMT, BENE_IP_DDCTBL_AMT, BENE_BLOOD_DDCTBL_AMT, BENE_PRMRY_PYR_AMT, DRG_OUTLIER_PMT_AMT, IP_DSPRPRTNT_SHR_AMT, IME_AMT, DRG_PRICE_AMT, PASS_THRU_AMT, TOT_PPS_CPTL_AMT, TOT_CHRG_AMT, TOT_CVR_CHRG_AMT, MDCR_PMT_AMT, ACMDTNS_TOT_CHRG_AMT, DPRTMNTL_TOT_CHRG_AMT, PRVT_ROOM_DAY_CNT, SEMIPRVT_ROOM_DAY_CNT, WARD_DAY_CNT, INTNSV_CARE_DAY_CNT, CRNRY_CARE_DAY_CNT, 